In [41]:
import pandas as pd
import country_converter as coco
from pathlib import Path

In [42]:
base=pd.read_csv('data/data_clean/country_year_sanctions_coflict.csv')
base["year"] = pd.to_datetime(base["year"].astype(str).str[:4], format="%Y")
df= pd.read_csv('data/data_raw/gdelt_conflict_1_0.csv')
df["Year"] = pd.to_datetime(df["Year"].astype(str), format="%Y")


In [43]:
df.sort_values(by=["Year", "CountryName"], inplace=True)
df

,Year,CountryCode,CountryName,SumEvents,TotalEvents,NormalizedEvents1000,EventRootCode,EventRootDescr,EventCode,EventDescr,GoldsteinScale,AvgNumMentions,SumNumMentions,AvgAvgTone
32,1979-01-01,AF,Afghanistan,2,3992,0.501002,18,ASSAULT,182,"Physically assault, not specified below",-9.5,3.000000,6,3.651636
47,1979-01-01,AF,Afghanistan,66,3992,16.533066,19,FIGHT,192,Occupy territory,-9.5,3.303030,218,4.597937
63,1979-01-01,AF,Afghanistan,33,3992,8.266533,18,ASSAULT,180,"Use unconventional violence, not specified below",-9.0,4.666667,154,4.550332
298,1979-01-01,AF,Afghanistan,44,3992,11.022044,19,FIGHT,193,Fight with small arms and light weapons,-10.0,4.522727,199,4.541173
350,1979-01-01,AF,Afghanistan,16,3992,4.008016,18,ASSAULT,181,"Abduct, hijack, or take hostage",-9.0,3.687500,59,5.075226
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175534,2021-01-01,ZI,Zimbabwe,1646,34384,47.871103,17,COERCE,173,"Arrest, detain, or charge with legal action",-5.0,8.866950,14595,-7.054593
175869,2021-01-01,ZI,Zimbabwe,31,34384,0.901582,17,COERCE,174,Expel or deport individuals,-5.0,6.322581,196,-4.004875
175876,2021-01-01,ZI,Zimbabwe,9,34384,0.261750,18,ASSAULT,185,Attempt to assassinate,-8.0,3.222222,29,-1.722121
176350,2021-01-01,ZI,Zimbabwe,76,34384,2.210330,18,ASSAULT,181,"Abduct, hijack, or take hostage",-9.0,7.815789,594,-7.104693


In [44]:

# drop exact duplicate rows (e.g. Burma/Myanmar same CountryCode, different CountryName)
df = df.drop_duplicates(subset=['Year', 'CountryCode', 'EventCode'])

# clean root descr into snake_case for column naming
root_map = {
    'COERCE': 'coerce',
    'ASSAULT': 'assault',
    'FIGHT': 'fight',
    'USE UNCONVENTIONAL MASS VIOLENCE': 'mass_violence',
}
df['root_clean'] = df['EventRootDescr'].map(root_map)

# Goldstein-weighted severity per event row (event count * intensity score)
df['goldstein_weighted'] = df['SumEvents'] * df['GoldsteinScale']

# 1. Normalized events by violence type, per country-year
by_type = (
    df.groupby(['Year', 'CountryCode', 'CountryName', 'root_clean'])['NormalizedEvents1000']
    .sum()
    .unstack('root_clean')
    .add_suffix('_norm1000')
    .reset_index()
)

# 2. Total normalized violence across all 4 categories, per country-year
total_norm = (
    df.groupby(['Year', 'CountryCode'])['NormalizedEvents1000']
    .sum()
    .reset_index()
    .rename(columns={'NormalizedEvents1000': 'total_violence_norm1000'})
)

# 3. Goldstein-weighted severity, by type
goldstein_by_type = (
    df.groupby(['Year', 'CountryCode'])['goldstein_weighted']  # placeholder, replaced below
    .sum()
    .reset_index()
)

goldstein_by_type = (
    df.groupby(['Year', 'CountryCode', 'root_clean'])['goldstein_weighted']
    .sum()
    .unstack('root_clean')
    .add_suffix('_goldstein')
    .reset_index()
)

# 4. Total Goldstein-weighted severity across all types
goldstein_total = (
    df.groupby(['Year', 'CountryCode'])['goldstein_weighted']
    .sum()
    .reset_index()
    .rename(columns={'goldstein_weighted': 'total_violence_goldstein'})
)

# merge everything into one wide panel
panel = by_type.merge(total_norm, on=['Year', 'CountryCode'])
panel = panel.merge(goldstein_by_type, on=['Year', 'CountryCode'])
panel = panel.merge(goldstein_total, on=['Year', 'CountryCode'])

# flip Goldstein sign so higher = more violent (consistent direction with death counts)
goldstein_cols = [c for c in panel.columns if 'goldstein' in c]
panel[goldstein_cols] = panel[goldstein_cols] * -1

# zero-fill missing category values (NaN here = zero events of that type that year, a true zero)
fill_cols = [c for c in panel.columns if 'norm1000' in c or 'goldstein' in c]
panel[fill_cols] = panel[fill_cols].fillna(0)

print(panel.shape)
print(panel.head())

(9749, 13)
        Year CountryCode           CountryName  assault_norm1000  \
0 1979-01-01          AE  United Arab Emirates          5.747126   
1 1979-01-01          AF           Afghanistan         16.783567   
2 1979-01-01          AG               Algeria          0.657030   
3 1979-01-01          AJ            Azerbaijan         27.491409   
4 1979-01-01          AL               Albania          2.985075   

   coerce_norm1000  fight_norm1000  mass_violence_norm1000  \
0        22.988506        8.620690                     0.0   
1        25.801603      134.018036                     0.0   
2        19.053876       16.425756                     0.0   
3        24.054983      106.529210                     0.0   
4         6.716418       44.029851                     0.0   

   total_violence_norm1000  assault_goldstein  coerce_goldstein  \
0                37.356322               18.0              40.0   
1               176.603206              612.0             573.4   
2     

In [45]:
panel

,Year,CountryCode,CountryName,assault_norm1000,coerce_norm1000,fight_norm1000,mass_violence_norm1000,total_violence_norm1000,assault_goldstein,coerce_goldstein,fight_goldstein,mass_violence_goldstein,total_violence_goldstein
0,1979-01-01,AE,United Arab Emirates,5.747126,22.988506,8.620690,0.000000,37.356322,18.0,40.0,30.0,0.0,88.0
1,1979-01-01,AF,Afghanistan,16.783567,25.801603,134.018036,0.000000,176.603206,612.0,573.4,5316.0,0.0,6501.4
2,1979-01-01,AG,Algeria,0.657030,19.053876,16.425756,0.000000,36.136662,9.0,149.2,248.0,0.0,406.2
3,1979-01-01,AJ,Azerbaijan,27.491409,24.054983,106.529210,0.000000,158.075601,72.0,35.0,309.0,0.0,416.0
4,1979-01-01,AL,Albania,2.985075,6.716418,44.029851,0.000000,53.731343,36.5,57.6,589.0,0.0,683.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9744,2021-01-01,WS,Samoa,10.602679,23.995536,21.763393,0.000000,56.361607,184.0,221.0,388.0,0.0,793.0
9745,2021-01-01,WZ,Swaziland,2.805049,55.399719,31.556802,0.000000,89.761571,37.5,419.8,446.5,0.0,903.8
9746,2021-01-01,YM,Yemen,17.036673,22.832654,137.382324,0.386399,177.638050,4479.0,3681.4,38921.0,110.0,47191.4
9747,2021-01-01,ZA,Zambia,11.153636,42.864954,29.852378,0.000000,83.870968,932.0,2029.0,2708.0,0.0,5669.0


In [46]:

names = panel['CountryName'].unique()
cc = coco.CountryConverter()

iso3 = cc.convert(names=list(names), to='ISO3', not_found=None)
gwc = cc.convert(names=list(names), to='GWcode', not_found=None)

crosswalk = pd.DataFrame({'CountryName': names, 'iso3c': iso3, 'gwcode': gwc})

crosswalk['iso3_valid'] = crosswalk['iso3c'].str.match(r'^[A-Z]{3}$', na=False)
crosswalk = crosswalk[crosswalk['iso3_valid']].drop(columns='iso3_valid')

# merge crosswalk into main panel FIRST
panel = panel.merge(crosswalk, on='CountryName', how='inner')

# THEN apply the gwcode fixes, now that iso3c/gwcode columns actually exist
gwcode_fixes = {
    'CZE': 316,
    'YEM': 678,
}
for iso, correct_code in gwcode_fixes.items():
    panel.loc[panel['iso3c'] == iso, 'gwcode'] = correct_code

print(panel.shape)
print(panel[['CountryName', 'iso3c', 'gwcode']].drop_duplicates().head(10))

Spratly Islands not found in regex
Wake Island not found in regex
Johnston Atoll not found in regex
Paracel Islands not found in regex
Palmyra Atoll not found in regex
Midway Islands not found in regex
Baker Island not found in regex
Howland Island not found in regex
Jan Mayen not found in regex
Coral Sea Islands not found in regex
Navassa Island not found in regex
Jarvis Island not found in regex
Tromelin Island not found in regex
Europa Island not found in regex
Clipperton Island not found in regex
Spratly Islands not found in regex
Wake Island not found in regex
Johnston Atoll not found in regex
Paracel Islands not found in regex
Palmyra Atoll not found in regex
Midway Islands not found in regex
Baker Island not found in regex
Howland Island not found in regex
Jan Mayen not found in regex
Coral Sea Islands not found in regex
Navassa Island not found in regex
Jarvis Island not found in regex
Tromelin Island not found in regex
Europa Island not found in regex
Clipperton Island not fou

(9629, 15)
            CountryName iso3c gwcode
0  United Arab Emirates   ARE    696
1           Afghanistan   AFG    700
2               Algeria   DZA    615
3            Azerbaijan   AZE    373
4               Albania   ALB    339
5               Armenia   ARM    371
6                Angola   AGO    540
7             Argentina   ARG    160
8             Australia   AUS    900
9               Austria   AUT    305


In [47]:
crosswalk

,CountryName,iso3c,gwcode
0,United Arab Emirates,ARE,696
1,Afghanistan,AFG,700
2,Algeria,DZA,615
3,Azerbaijan,AZE,373
4,Albania,ALB,339
...,...,...,...
242,British Indian Ocean Territory,IOT,NaN
243,Niue,NIU,NaN
244,Pitcairn Islands,PCN,NaN
251,Bouvet Island,BVT,NaN


In [48]:
panel

,Year,CountryCode,CountryName,assault_norm1000,coerce_norm1000,fight_norm1000,mass_violence_norm1000,total_violence_norm1000,assault_goldstein,coerce_goldstein,fight_goldstein,mass_violence_goldstein,total_violence_goldstein,iso3c,gwcode
0,1979-01-01,AE,United Arab Emirates,5.747126,22.988506,8.620690,0.000000,37.356322,18.0,40.0,30.0,0.0,88.0,ARE,696
1,1979-01-01,AF,Afghanistan,16.783567,25.801603,134.018036,0.000000,176.603206,612.0,573.4,5316.0,0.0,6501.4,AFG,700
2,1979-01-01,AG,Algeria,0.657030,19.053876,16.425756,0.000000,36.136662,9.0,149.2,248.0,0.0,406.2,DZA,615
3,1979-01-01,AJ,Azerbaijan,27.491409,24.054983,106.529210,0.000000,158.075601,72.0,35.0,309.0,0.0,416.0,AZE,373
4,1979-01-01,AL,Albania,2.985075,6.716418,44.029851,0.000000,53.731343,36.5,57.6,589.0,0.0,683.1,ALB,339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9624,2021-01-01,WS,Samoa,10.602679,23.995536,21.763393,0.000000,56.361607,184.0,221.0,388.0,0.0,793.0,WSM,990
9625,2021-01-01,WZ,Swaziland,2.805049,55.399719,31.556802,0.000000,89.761571,37.5,419.8,446.5,0.0,903.8,SWZ,572
9626,2021-01-01,YM,Yemen,17.036673,22.832654,137.382324,0.386399,177.638050,4479.0,3681.4,38921.0,110.0,47191.4,YEM,678
9627,2021-01-01,ZA,Zambia,11.153636,42.864954,29.852378,0.000000,83.870968,932.0,2029.0,2708.0,0.0,5669.0,ZMB,551


In [49]:
base_gdelt=pd.merge(base, panel, how='left',
    left_on=['year','gwcode'], right_on=['Year',  'gwcode'])

base_gdelt

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c_x,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,coerce_norm1000,fight_norm1000,mass_violence_norm1000,total_violence_norm1000,assault_goldstein,coerce_goldstein,fight_goldstein,mass_violence_goldstein,total_violence_goldstein,iso3c_y
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,73.820556,45.492839,0.0,132.476832,2297.0,7162.6,8613.0,0.0,18072.6,WSM
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,54.104683,27.988981,0.0,90.578512,715.0,2527.8,2529.0,0.0,5771.8,WSM
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,23.995536,21.763393,0.0,56.361607,184.0,221.0,388.0,0.0,793.0,WSM
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:

numeric_cols=[
 'assault_norm1000',
 'coerce_norm1000',
 'fight_norm1000',
 'mass_violence_norm1000',
 'total_violence_norm1000',
 'assault_goldstein',
 'coerce_goldstein',
 'fight_goldstein',
 'mass_violence_goldstein',
 'total_violence_goldstein']

In [51]:
mask = base_gdelt['year'] > pd.Timestamp('1978-01-01')
base_gdelt.loc[mask, numeric_cols] = base_gdelt.loc[mask, numeric_cols].fillna(0)
base_gdelt

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c_x,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,coerce_norm1000,fight_norm1000,mass_violence_norm1000,total_violence_norm1000,assault_goldstein,coerce_goldstein,fight_goldstein,mass_violence_goldstein,total_violence_goldstein,iso3c_y
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,73.820556,45.492839,0.0,132.476832,2297.0,7162.6,8613.0,0.0,18072.6,WSM
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,54.104683,27.988981,0.0,90.578512,715.0,2527.8,2529.0,0.0,5771.8,WSM
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,23.995536,21.763393,0.0,56.361607,184.0,221.0,388.0,0.0,793.0,WSM
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN


In [52]:
base_gdelt

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c_x,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,coerce_norm1000,fight_norm1000,mass_violence_norm1000,total_violence_norm1000,assault_goldstein,coerce_goldstein,fight_goldstein,mass_violence_goldstein,total_violence_goldstein,iso3c_y
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,73.820556,45.492839,0.0,132.476832,2297.0,7162.6,8613.0,0.0,18072.6,WSM
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,54.104683,27.988981,0.0,90.578512,715.0,2527.8,2529.0,0.0,5771.8,WSM
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,23.995536,21.763393,0.0,56.361607,184.0,221.0,388.0,0.0,793.0,WSM
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,NaN


In [53]:
output_dir = Path('data/data_clean')
output_dir.mkdir(parents=True, exist_ok=True)

base_gdelt.to_csv(output_dir / 'country_year_sanctions_coflict_gdelt.csv', index=False)

check data

In [54]:
base_gdelt=pd.read_csv('data/data_clean/country_year_sanctions_coflict_gdelt.csv')

In [55]:
cols=[base_gdelt.columns]
cols

[Index(['gwcode', 'gw_name', 'microstate', 'year', 'ccode', 'gwc', 'iso3c_x',
        'successor_iso3c', 'n_conflicts_all', 'n_conflicts_intra',
        ...
        'coerce_norm1000', 'fight_norm1000', 'mass_violence_norm1000',
        'total_violence_norm1000', 'assault_goldstein', 'coerce_goldstein',
        'fight_goldstein', 'mass_violence_goldstein',
        'total_violence_goldstein', 'iso3c_y'],
       dtype='str', length=123)]

In [56]:
# assuming df has columns: gw_name, year (equivalent to your panel index)
tab = base_gdelt.groupby(["gw_name", "year"]).size()

# equivalent of tab[tab > 1]
dups = tab[tab > 1]
print(dups)

Series([], dtype: int64)


In [57]:
# reset_index to turn the grouped counts back into a regular dataframe
dup_pairs = dups.reset_index()

# pull the original rows that correspond to those duplicated pairs
dup_rows = base_gdelt.merge(dup_pairs[["gw_name", "year"]], on=["gw_name", "year"])
print(dup_rows.sort_values(["gw_name", "year"]))

Empty DataFrame
Columns: [gwcode, gw_name, microstate, year, ccode, gwc, iso3c_x, successor_iso3c, n_conflicts_all, n_conflicts_intra, n_conflicts_inter, n_conflicts_extra, euds, aeuds, polity2, v2x_polyarchy, mrgdppc, sd_mrgdppc, pwtrgdp, sd_pwtrgdp, pwtpop, sd_pwtpop, num_sanctioners, n_arms_received, n_military_received, n_trade_received, n_financial_received, n_travel_received, n_other_received, num_targets_sanctioned, n_arms_sent, n_military_sent, n_trade_sent, n_financial_sent, n_travel_sent, n_other_sent, country, country_id, region, govt_name, sb_exist, sb_intrastate_exist, sb_interstate_exist, ns_exist, os_exist, sb_intrastate_govt_inv_incomp, sb_interstate_govt_inv_incomp, os_govt_inv, os_any_govt_inv, sb_dyad_count, sb_intrastate_dyad_count, sb_interstate_dyad_count, ns_dyad_count, os_dyad_count, cumulative_total_deaths_parties_in_orgvio, cumulative_total_deaths_civilians_in_orgvio, cumulative_total_deaths_unknown_in_orgvio, cumulative_total_deaths_in_orgvio_best, sb_deaths_

In [58]:
base=pd.read_csv('data/data_clean/final.csv')
base

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,euds,aeuds,polity2,v2x_polyarchy,mrgdppc,sd_mrgdppc,pwtrgdp,sd_pwtrgdp,pwtpop,sd_pwtpop
0,2,United States of America,0,1946,2.0,USA,USA,NaN,0,0,...,1.293985,0.486256,9.0,0.603,15916.605087,2494.189575,2.565007e+06,1.075414e+06,142.942214,9.874863
1,2,United States of America,0,1947,2.0,USA,USA,NaN,0,0,...,1.308359,0.500630,9.0,0.607,15691.605157,2443.671758,2.524909e+06,1.097010e+06,145.613447,10.302882
2,2,United States of America,0,1948,2.0,USA,USA,NaN,0,0,...,1.343539,0.535809,9.0,0.599,15634.165601,2397.095016,2.621770e+06,1.143350e+06,148.443589,10.571322
3,2,United States of America,0,1949,2.0,USA,USA,NaN,0,0,...,1.330836,0.523106,9.0,0.580,15774.791549,2415.503157,2.724009e+06,1.187962e+06,150.554205,10.712760
4,2,United States of America,0,1950,2.0,USA,USA,NaN,1,0,...,1.354015,0.546286,9.0,0.585,16610.071657,2605.983858,2.831568e+06,1.207133e+06,153.549515,10.469628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019,990.0,WSM,WSM,NaN,0,0,...,0.846640,0.038911,NaN,NaN,5280.997786,7602.198097,1.161916e+03,1.830815e+03,0.196232,0.015295
12306,990,Samoa/Western Samoa,1,2020,990.0,WSM,WSM,NaN,0,0,...,0.988814,0.181084,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12307,990,Samoa/Western Samoa,1,2021,990.0,WSM,WSM,NaN,0,0,...,1.327329,0.519600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12308,990,Samoa/Western Samoa,1,2022,990.0,WSM,WSM,NaN,0,0,...,1.008024,0.200294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
